# Modul 3 - Hand & Pose Tracking (Google Colab)

**Fokus:** cara kerja landmark dan tracking
**Hands-on:** virtual painter atau kontrol volume dengan jari
**Target:** peserta dapat melacak tangan dan membuat minimal satu interaksi yang dipicu gesture.

### Cara kerja di Colab
Colab berjalan di server, bukan di laptopmu, jadi `cv2.VideoCapture(0)` dan `cv2.imshow` **tidak bisa** dipakai. Gantinya:

- **Webcam live:** kamera dibuka oleh browser (JavaScript), tiap frame dikirim ke Python, diproses MediaPipe, lalu hasilnya ditampilkan kembali. Browser akan meminta izin kamera. FPS-nya lebih rendah (sekitar 5-10) daripada versi lokal.
- **File video:** upload video, diproses, lalu hasilnya diputar di notebook. Cocok sebagai cadangan kalau webcam bermasalah.

Urutan: Setup -> Kode inti -> Tracking dasar -> Virtual Painter -> Kontrol Volume -> (opsional) Video file.

> Gunakan **Chrome** dan izinkan akses kamera. Untuk berhenti saat live, **klik video** atau stop sel (tombol stop).

## 1. Setup

In [ ]:
!pip install -q mediapipe opencv-python-headless

In [ ]:
import os
import urllib.request
import mediapipe as mp
print("mediapipe", mp.__version__)

# Model hand landmarker resmi dari Google (dipakai oleh MediaPipe Tasks API)
MODEL_PATH = "hand_landmarker.task"
MODEL_URL = ("https://storage.googleapis.com/mediapipe-models/hand_landmarker/"
             "hand_landmarker/float16/1/hand_landmarker.task")
if not os.path.exists(MODEL_PATH):
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
print("model siap:", os.path.getsize(MODEL_PATH) // 1024, "KB")

## 2. Kode inti

Satu kelas `HandApp` dengan tiga mode: `tracking`, `painter`, `volume`. Dipakai oleh webcam live maupun file video, jadi logikanya cuma ditulis sekali.

Deteksi memakai **MediaPipe Tasks `HandLandmarker`** (API terbaru, karena `mp.solutions` yang lama sudah dihapus di mediapipe versi baru). Mode `VIDEO` membuat tangan dilacak antar-frame, bukan dideteksi ulang dari nol tiap frame.

MediaPipe Hands mendeteksi **21 landmark** per tangan: 0 = pergelangan, 4 = ujung jempol, 8 = ujung telunjuk, 12 = ujung tengah, 16 = ujung manis, 20 = ujung kelingking.

Alur tiap frame: mirror -> BGR ke RGB -> `hands.process()` -> gambar landmark -> baca gesture.

In [ ]:
import math
import time
from base64 import b64decode, b64encode

import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

# Pasangan landmark yang dihubungkan garis (kerangka tangan)
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),          # jempol
    (0, 5), (5, 6), (6, 7), (7, 8),          # telunjuk
    (5, 9), (9, 10), (10, 11), (11, 12),     # tengah
    (9, 13), (13, 14), (14, 15), (15, 16),   # manis
    (13, 17), (17, 18), (18, 19), (19, 20),  # kelingking
    (0, 17),                                 # sisi telapak
]


def draw_hand(frame, pts):
    for a, b in HAND_CONNECTIONS:
        cv2.line(frame, pts[a], pts[b], (255, 255, 255), 2)
    for p in pts:
        cv2.circle(frame, p, 4, (0, 0, 255), cv2.FILLED)


# Urutan hasil fingers_up(): [jempol, telunjuk, tengah, manis, kelingking]
FINGER_NAMES = ["Jempol", "Telunjuk", "Tengah", "Manis", "Kelingking"]
PAINT_COLORS = [((255, 120, 0), "Biru"), ((0, 200, 0), "Hijau"),
                ((0, 0, 255), "Merah"), ((0, 255, 255), "Kuning")]


def to_pixels(lms, w, h):
    # Landmark MediaPipe berskala 0-1 -> koordinat piksel (x, y)
    return [(int(lm.x * w), int(lm.y * h)) for lm in lms]


def fingers_up(lm, label):
    # Return 5 boolean [jempol, telunjuk, tengah, manis, kelingking].
    # lm = daftar 21 landmark. Asumsi: frame sudah di-mirror dan telapak menghadap kamera.
    # Selain jempol: ujung (tip) lebih tinggi dari sendi PIP -> terangkat.
    # Jempol: dibandingkan di sumbu X, tergantung tangan kiri/kanan.
    up = []
    if label == "Right":
        up.append(lm[4].x < lm[3].x)
    else:
        up.append(lm[4].x > lm[3].x)
    for tip in (8, 12, 16, 20):
        up.append(lm[tip].y < lm[tip - 2].y)
    return up


class HandApp:
    # Kontrol volume: rasio jarak (jempol-telunjuk) / ukuran telapak
    RATIO_MIN = 0.25   # jari menempel -> 0%
    RATIO_MAX = 1.40   # jari terbuka lebar -> 100%
    SMOOTH = 0.2       # makin kecil makin halus tapi makin lambat

    def __init__(self, mode="tracking", clear_hold=10, flip=True):
        assert mode in ("tracking", "painter", "volume"), "mode: tracking / painter / volume"
        self.mode = mode
        self.flip = flip
        options = vision.HandLandmarkerOptions(
            base_options=mp_python.BaseOptions(model_asset_path=MODEL_PATH),
            running_mode=vision.RunningMode.VIDEO,  # mode video = antar-frame dilacak (tracking)
            num_hands=2 if mode == "tracking" else 1,
            min_hand_detection_confidence=0.7,
            min_hand_presence_confidence=0.5,
            min_tracking_confidence=0.6,
        )
        self.landmarker = vision.HandLandmarker.create_from_options(options)
        self.ts_ms = 0  # timestamp harus selalu naik
        # state painter
        self.canvas = None
        self.prev_pt = None
        self.color_idx = 0
        self.clear_hold = clear_hold   # jumlah frame telapak ditahan untuk hapus
        self.clear_counter = 0
        self.color_cooldown = 0
        # state volume
        self.vol = 0.0
        # fps
        self.prev_time = 0
        self.fps = 0.0

    def close(self):
        self.landmarker.close()

    def detect(self, frame):
        # Return list tangan: [(label, [21 landmark]), ...]. label = "Left" / "Right"
        rgb = np.ascontiguousarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        self.ts_ms = max(self.ts_ms + 1, int(time.time() * 1000))
        res = self.landmarker.detect_for_video(mp_image, self.ts_ms)
        return [(handed[0].category_name, lms)
                for lms, handed in zip(res.hand_landmarks, res.handedness)]

    def process(self, frame):
        # Return (frame_hasil, teks_status)
        if self.flip:
            frame = cv2.flip(frame, 1)
        h, w, _ = frame.shape
        result = self.detect(frame)

        if self.mode == "tracking":
            status = self._tracking(frame, result, w, h)
        elif self.mode == "painter":
            status = self._painter(frame, result, w, h)
        else:
            status = self._volume(frame, result, w, h)

        now = time.time()
        if self.prev_time:
            inst = 1 / max(now - self.prev_time, 1e-6)
            self.fps = inst if self.fps == 0 else 0.9 * self.fps + 0.1 * inst
        self.prev_time = now
        cv2.putText(frame, f"FPS: {int(self.fps)}", (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        return frame, status

    # ---- mode 1: tracking dasar ----
    def _tracking(self, frame, result, w, h):
        if not result:
            return "Tidak ada tangan"
        info = []
        for label, hand_lms in result:
            pts = to_pixels(hand_lms, w, h)
            draw_hand(frame, pts)
            up = fingers_up(hand_lms, label)
            cv2.circle(frame, pts[8], 10, (0, 255, 0), cv2.FILLED)  # ujung telunjuk
            x0, y0 = pts[0]
            cv2.putText(frame, f"{label}: {sum(up)} jari", (max(x0 - 40, 0), min(y0 + 35, h - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
            names = [n for n, u in zip(FINGER_NAMES, up) if u]
            info.append(f"{label}: {', '.join(names) if names else '-'}")
        return " | ".join(info)

    # ---- mode 2: virtual painter ----
    def _painter(self, frame, result, w, h):
        if self.canvas is None or self.canvas.shape != frame.shape:
            self.canvas = np.zeros_like(frame)
        color, color_name = PAINT_COLORS[self.color_idx]
        status = "Tidak ada tangan"
        if self.color_cooldown > 0:
            self.color_cooldown -= 1

        if result:
            label, hand_lms = result[0]
            pts = to_pixels(hand_lms, w, h)
            draw_hand(frame, pts)
            thumb, index, middle, ring, pinky = fingers_up(hand_lms, label)
            tip = pts[8]

            if all([thumb, index, middle, ring, pinky]):
                # Telapak terbuka ditahan -> hapus semua
                status = f"Hapus semua... {self.clear_counter * 100 // self.clear_hold}%"
                self.clear_counter += 1
                self.prev_pt = None
                if self.clear_counter >= self.clear_hold:
                    self.canvas[:] = 0
                    self.clear_counter = 0
            else:
                self.clear_counter = 0
                if index and middle and ring and not pinky:
                    # 3 jari -> ganti warna (ada jeda supaya tidak berganti terus)
                    status = "Ganti warna"
                    self.prev_pt = None
                    if self.color_cooldown == 0:
                        self.color_idx = (self.color_idx + 1) % len(PAINT_COLORS)
                        self.color_cooldown = 15
                elif index and middle and not ring and not pinky:
                    status = "Pindah (pen diangkat)"
                    self.prev_pt = None
                    cv2.circle(frame, tip, 12, color, 2)
                elif index and not middle and not ring and not pinky:
                    status = "Menggambar"
                    cv2.circle(frame, tip, 8, color, cv2.FILLED)
                    if self.prev_pt is not None:
                        cv2.line(self.canvas, self.prev_pt, tip, color, 8)
                    self.prev_pt = tip
                else:
                    status = "Siap (angkat telunjuk)"
                    self.prev_pt = None
        else:
            self.prev_pt = None
            self.clear_counter = 0

        # Tempelkan canvas ke frame (piksel yang bukan hitam)
        mask = self.canvas.any(axis=2)
        frame[mask] = self.canvas[mask]
        color, color_name = PAINT_COLORS[self.color_idx]
        cv2.putText(frame, f"Warna: {color_name}", (10, 55),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
        return f"{status} | Warna: {color_name}"

    # ---- mode 3: kontrol volume ----
    def _volume(self, frame, result, w, h):
        status = "Tidak ada tangan"
        if result:
            _, hand_lms = result[0]
            pts = to_pixels(hand_lms, w, h)
            draw_hand(frame, pts)
            thumb_tip, index_tip = pts[4], pts[8]
            palm_size = math.dist(pts[0], pts[5])  # pergelangan -> pangkal telunjuk
            if palm_size > 0:
                ratio = math.dist(thumb_tip, index_tip) / palm_size
                target = float(np.interp(ratio, [self.RATIO_MIN, self.RATIO_MAX], [0, 100]))
                self.vol += (target - self.vol) * self.SMOOTH  # smoothing (EMA)
            cv2.line(frame, thumb_tip, index_tip, (255, 0, 255), 3)
            cv2.circle(frame, thumb_tip, 10, (255, 0, 255), cv2.FILLED)
            cv2.circle(frame, index_tip, 10, (255, 0, 255), cv2.FILLED)
            status = "Atur volume"

        # Bar volume
        bar_top, bar_bottom = int(h * 0.2), int(h * 0.8)
        fill = int(np.interp(self.vol, [0, 100], [bar_bottom, bar_top]))
        cv2.rectangle(frame, (30, bar_top), (65, bar_bottom), (255, 255, 255), 2)
        cv2.rectangle(frame, (30, fill), (65, bar_bottom), (0, 255, 0), cv2.FILLED)
        cv2.putText(frame, f"{int(self.vol)}%", (20, bar_bottom + 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
        return f"{status} | Volume: {int(self.vol)}%"

## 3. Webcam live

Kode di bawah membuka kamera lewat browser (JavaScript), mengirim tiap frame ke Python, lalu menampilkan hasil proses.
Fungsi `run_live(mode)` dipakai untuk ketiga langkah hands-on.

In [ ]:
from IPython.display import Javascript, display, HTML
from google.colab.output import eval_js

def video_stream():
    js = Javascript('''
    var video; var div = null; var stream; var captureCanvas; var imgElement; var labelElement;
    var pendingResolve = null; var shutdown = false;

    function removeDom() {
      if (stream) { stream.getVideoTracks()[0].stop(); }
      if (video) { video.remove(); }
      if (div) { div.remove(); }
      video = null; div = null; stream = null; imgElement = null; captureCanvas = null; labelElement = null;
    }

    function onAnimationFrame() {
      if (!shutdown) { window.requestAnimationFrame(onAnimationFrame); }
      if (pendingResolve) {
        var result = "";
        if (!shutdown) {
          captureCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
          result = captureCanvas.toDataURL('image/jpeg', 0.8);
        }
        var lp = pendingResolve;
        pendingResolve = null;
        lp(result);
      }
    }

    async function createDom() {
      if (div !== null) { return stream; }
      div = document.createElement('div');
      div.style.border = '2px solid black';
      div.style.padding = '3px';
      div.style.width = '100%';
      div.style.maxWidth = '640px';
      document.body.appendChild(div);

      var modelOut = document.createElement('div');
      modelOut.innerHTML = '<b>Status:</b> ';
      labelElement = document.createElement('span');
      labelElement.innerText = 'menyiapkan...';
      modelOut.appendChild(labelElement);
      div.appendChild(modelOut);

      var hint = document.createElement('div');
      hint.innerText = 'Klik video untuk berhenti';
      hint.style.fontSize = '12px';
      hint.style.color = '#666';
      div.appendChild(hint);

      video = document.createElement('video');
      video.style.display = 'block';
      video.width = div.clientWidth - 6;
      video.setAttribute('playsinline', '');
      video.onclick = function() { shutdown = true; };
      stream = await navigator.mediaDevices.getUserMedia({video: {facingMode: 'user'}});
      div.appendChild(video);

      imgElement = document.createElement('img');
      imgElement.style.position = 'absolute';
      imgElement.style.zIndex = 1;
      imgElement.onclick = function() { shutdown = true; };
      div.appendChild(imgElement);

      video.srcObject = stream;
      await video.play();

      captureCanvas = document.createElement('canvas');
      captureCanvas.width = 640;
      captureCanvas.height = 480;
      window.requestAnimationFrame(onAnimationFrame);
      return stream;
    }

    async function stream_frame(label, imgData) {
      if (shutdown) { removeDom(); shutdown = false; return ''; }
      stream = await createDom();
      if (label != '') { labelElement.innerText = label; }
      if (imgData != '') {
        var rect = video.getClientRects()[0];
        imgElement.style.top = rect.top + 'px';
        imgElement.style.left = rect.left + 'px';
        imgElement.style.width = rect.width + 'px';
        imgElement.style.height = rect.height + 'px';
        imgElement.src = imgData;
      }
      var result = await new Promise(function(resolve, reject) { pendingResolve = resolve; });
      if (shutdown) { removeDom(); shutdown = false; return ''; }
      return {'img': result};
    }

    function stream_stop() {
      removeDom();
      shutdown = false;
      return '';
    }
    ''')
    display(js)


def video_frame(label, overlay):
    return eval_js('stream_frame("{}", "{}")'.format(label, overlay))


def js_to_image(data_url):
    img_bytes = b64decode(data_url.split(",")[1])
    return cv2.imdecode(np.frombuffer(img_bytes, dtype=np.uint8), cv2.IMREAD_COLOR)


def image_to_uri(img):
    ok, buf = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 70])
    return "data:image/jpeg;base64," + b64encode(buf.tobytes()).decode()


def run_live(mode="tracking", max_seconds=180):
    # Jalankan webcam live. Berhenti: klik video, stop sel, atau otomatis setelah max_seconds.
    app = HandApp(mode=mode, clear_hold=8)  # fps rendah di Colab, jadi tahan telapak lebih singkat
    video_stream()
    label, overlay = "menyiapkan kamera...", ""
    t0 = time.time()
    try:
        while time.time() - t0 < max_seconds:
            reply = video_frame(label, overlay)
            if not reply:
                break
            frame = js_to_image(reply["img"])
            out, label = app.process(frame)
            overlay = image_to_uri(out)
    except KeyboardInterrupt:
        pass
    finally:
        try:
            eval_js("stream_stop()")
        except Exception:
            pass
        app.close()
    print("Selesai.")

## 4. Hand Tracking Dasar

Tampilkan tangan ke kamera. Perhatikan 21 titik landmark, jumlah jari yang terangkat, dan nama jarinya di bagian **Status**.

In [ ]:
run_live("tracking")

## 5. Virtual Painter

Menggambar di udara dengan jari telunjuk.

| Gesture | Aksi |
|---|---|
| Telunjuk saja | **Menggambar** |
| Telunjuk + tengah | Pindah posisi (pen diangkat) |
| Telunjuk + tengah + manis | **Ganti warna** (biru, hijau, merah, kuning) |
| Telapak terbuka (5 jari), ditahan sebentar | **Hapus semua** |

Di Colab tidak ada tombol keyboard untuk jendela video, jadi ganti warna dan hapus semuanya lewat gesture.

In [ ]:
run_live("painter")

## 6. Kontrol Volume dengan Jari

Jarak ujung jempol (landmark 4) ke ujung telunjuk (landmark 8) dipetakan ke 0-100%.
Jarak **dinormalisasi** dengan ukuran telapak (landmark 0 ke 5), jadi hasilnya konsisten walau tangan dekat atau jauh dari kamera. Nilai dihaluskan dengan smoothing (EMA).

Di Colab ini **simulasi** (bar volume di layar), karena Colab tidak bisa mengubah volume komputermu.

Kalau hasilnya kurang pas, atur `HandApp.RATIO_MIN`, `HandApp.RATIO_MAX`, dan `HandApp.SMOOTH` (kalibrasi), lalu jalankan ulang.

In [ ]:
# Opsional: kalibrasi
# HandApp.RATIO_MIN = 0.25
# HandApp.RATIO_MAX = 1.40
# HandApp.SMOOTH = 0.2

run_live("volume")

## 7. (Cadangan) Proses file video

Kalau webcam tidak bisa dipakai, upload video yang berisi tangan (rekaman dari HP atau laptop juga boleh), lalu proses dengan mode yang sama.

In [ ]:
import subprocess
from google.colab import files


def process_video(path, mode="tracking", max_seconds=20, out_path="hasil.mp4"):
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise RuntimeError("Video tidak bisa dibuka: " + str(path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    app = HandApp(mode=mode, clear_hold=max(int(fps * 0.8), 5))
    writer, n = None, 0
    max_frames = int(fps * max_seconds)
    try:
        while n < max_frames:
            ok, frame = cap.read()
            if not ok:
                break
            h, w = frame.shape[:2]
            if w > 640:  # kecilkan biar cepat
                frame = cv2.resize(frame, (640, int(h * 640 / w)))
            out, _ = app.process(frame)
            if writer is None:
                oh, ow = out.shape[:2]
                writer = cv2.VideoWriter("tmp_raw.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (ow, oh))
            writer.write(out)
            n += 1
    finally:
        cap.release()
        if writer is not None:
            writer.release()
        app.close()
    # Konversi ke H.264 supaya bisa diputar di browser
    subprocess.run(["ffmpeg", "-y", "-i", "tmp_raw.mp4", "-vcodec", "libx264",
                    "-pix_fmt", "yuv420p", "-loglevel", "error", out_path], check=True)
    print(f"{n} frame diproses -> {out_path}")
    return out_path


def show_video(path):
    data = b64encode(open(path, "rb").read()).decode()
    display(HTML('<video width="640" controls><source src="data:video/mp4;base64,' + data + '" type="video/mp4"></video>'))


uploaded = files.upload()          # pilih file video
video_path = next(iter(uploaded))
MODE = "tracking"                  # ganti: "tracking" / "painter" / "volume"
result_path = process_video(video_path, mode=MODE)
show_video(result_path)
# files.download(result_path)      # hapus tanda # kalau mau mengunduh hasilnya

## Troubleshooting

- **Kamera tidak muncul / izin ditolak:** klik ikon gembok di address bar, izinkan Kamera untuk `colab.research.google.com`, lalu jalankan ulang sel. Pakai Chrome.
- **Gagal mengunduh model (`hand_landmarker.task`):** cek koneksi internet runtime Colab, lalu jalankan ulang sel Setup.
- **Error soal versi mediapipe / numpy:** *Runtime > Restart session*, lalu jalankan ulang dari awal.
- **Terasa lambat (FPS rendah):** normal di Colab karena tiap frame bolak-balik browser-server. Pencahayaan terang dan latar polos membantu deteksi.
- **Kamera masih menyala setelah berhenti:** jalankan `eval_js("stream_stop()")` atau refresh halaman.
- **Jempol terbaca terbalik:** pastikan telapak menghadap kamera.